# 04 · Build / refine a dashboard
**Session 3 · step 4 of 4 · ~15 min**

🧠 **The idea.** Genie answers *ad-hoc* questions; a dashboard answers the *standing* ones
at a glance and can be shared/scheduled. Same governed gold tables underneath — so the
dashboard, the Genie space, and the SQL all agree by construction.

🏢 **Why FHLB-Topeka cares.** A concentration + collateral + housing view is the kind of
monitoring the Bank wants standing, governed, and shareable — not rebuilt in a spreadsheet
each week.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG (1 of 2)  —  pick your catalog, THEN run the next cell
# ============================================================
# Mode A (live, our workspace):  serverless_stable_6fhczt_catalog  (the default)
# Mode B (portable / Free Edition): type the catalog 00_LOAD_DATA used
#   (the one you created, or an existing one you loaded into). Same value in every module.
# This cell only creates the picker at the top of the notebook.
dbutils.widgets.text("catalog", "serverless_stable_6fhczt_catalog", "Catalog")
print("↑ Set the 'Catalog' widget at the top of the notebook, then run the next cell.")

In [ ]:
# ---- WORKSHOP CONFIG (2 of 2)  —  apply the selected catalog ----
CATALOG = dbutils.widgets.get("catalog").strip()
assert CATALOG, "Set the 'Catalog' widget at the top of the notebook, then re-run this cell."

GOLD   = f"{CATALOG}.fhlb_gold"     # governed, analyst-ready data products (read-only)
SILVER = f"{CATALOG}.fhlb_silver"   # cleaned/typed layer (we use the HPI time series here)

spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog: {CATALOG}  ·  gold: {GOLD}")

## Step 1 — Start an AI/BI dashboard (name it after yourself)
1. Left nav → **Dashboards** → **Create dashboard**.
2. **Name it `firstname-lastname FHLB Monitor`.**
3. You'll add a **dataset** (a SQL query) and then **visualizations** on top of it.

## Step 2 — Dataset: the concentration view
In the dashboard's **Data** tab, add a dataset with this query (this is your module-02
Beat 1, lightly shaped for charting):

In [ ]:
%sql
SELECT member_name, state,
       ROUND(total_outstanding_par/1e6, 1) AS outstanding_$m,
       ROUND(share_of_advance_book*100, 1) AS pct_of_book
FROM fhlb_gold.portfolio_concentration
ORDER BY total_outstanding_par DESC

## Step 3 — Add three visualizations
On the **Canvas**:
1. **Bar chart** — `member_name` (x) by `outstanding_$m` (y). Sort descending. This *is* the
   concentration story: the first bar (Midwest Savings Bank) towers.
2. **Counter / KPI** — the top member's `pct_of_book` (≈ **24.3%**). Add a second counter for
   total book (**$1,382M**).
3. **Housing line chart** — add a second dataset (the district HPI trend from module 02, Beat 4)
   and plot `avg_hpi` over `year`. Label the newest partial period so it doesn't read as a cliff.

💡 **Ask Genie (tie it together).** Anything you can chart, you can also ask your space — a
dashboard tile and a Genie answer on the same governed table will match. That's the analyst
enablement promise in one sentence.

## Step 4 — A geo/aggregated angle (optional)
If you add a map, **aggregate to one point per state**, not per member/row — a per-row map
chokes and misleads. Group advances by `state` and size/color by the total. (You already have
`state` on `portfolio_concentration`.)

In [ ]:
%sql
-- dataset for a state-level map or bar: advances by state
SELECT state, ROUND(SUM(total_outstanding_par)/1e6, 1) AS outstanding_$m,
       COUNT(*) AS members
FROM fhlb_gold.portfolio_concentration
GROUP BY state
ORDER BY outstanding_$m DESC

## 🧑‍💻 Your Turn
- Add a **collateral tile**: a counter for the count of undercollateralized members (should be **1**)
  and one for members with **stale valuations** (**7**). (Source: `member_collateral_capacity`.)
- **Publish** your dashboard and open the viewer — note it respects UC permissions, like Genie.

## ⚠️ Fallback
If a viz won't render, check the dataset query runs on its own in the SQL editor first — a
dashboard viz is only ever as good as its dataset query.

## 🌟 Optional / Going deeper
- Add an **MPF-by-product** bar (delinquency_pct by product) — the Xtra vs 35 spread.
- Add a **filter** on `state` and watch every tile update together.

---
### ✅ Done — you've gone governed-data → SQL → your own Genie space → dashboard.
**Wrap:** bring your Genie space + dashboard to the Session 5 discussion — they're inputs to
the use-case prioritization.